***POSITIONAL ENCODING & SELF ATTENTION***

In [12]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    MultiHeadAttention
)

In [13]:
sentence = ["I Love Deep Learning"]

print("Sentence:")
print(sentence)

Sentence:
['I Love Deep Learning']


***TOKENIZATION***

In [14]:
vectorizer = TextVectorization(
    output_mode="int",
    output_sequence_length=4
)

vectorizer.adapt(sentence)

tokens = vectorizer(sentence)

print("Vocabulary:")
print(vectorizer.get_vocabulary())

print("\nTokens:")
print(tokens.numpy())

Vocabulary:
['', '[UNK]', np.str_('love'), np.str_('learning'), np.str_('i'), np.str_('deep')]

Tokens:
[[4 2 5 3]]


***WORD EMBEDDINGS***

In [15]:
embedding_dim = 8

embedding_layer = Embedding(
    input_dim=len(vectorizer.get_vocabulary()),
    output_dim=embedding_dim
)

word_embeddings = embedding_layer(tokens)

print("Word Embeddings Shape:")
print(word_embeddings.shape)

print("\nWord Embeddings:")
print(word_embeddings.numpy())

Word Embeddings Shape:
(1, 4, 8)

Word Embeddings:
[[[-0.01361709  0.03724055 -0.03849368 -0.03118557  0.04991556
   -0.02148163 -0.02508724 -0.0084397 ]
  [-0.00453042  0.00099651  0.04099876  0.00815914 -0.01433401
   -0.04442031 -0.03799991 -0.04821364]
  [-0.0380661   0.03820462  0.02477906 -0.03745335  0.03697581
   -0.00492955 -0.02669394  0.02621701]
  [-0.03321864  0.01387859  0.03140593  0.02678398  0.03579182
    0.02532103 -0.04740982 -0.01098639]]]


***POSITIONAL ENCODING FUNCTION***

In [16]:
def positional_encoding(max_position, d_model):
    positions = np.arange(max_position)[:, np.newaxis]
    dimensions = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(
        10000,
        (2 * (dimensions // 2)) / d_model
    )
    angle_rads = positions * angle_rates
    PE = np.zeros((max_position, d_model))
    PE[:, 0::2] = np.sin(angle_rads[:, 0::2])
    PE[:, 1::2] = np.cos(angle_rads[:, 1::2])
    return tf.cast(PE, dtype=tf.float32)

***GENERAL POSITIONAL ENCODING***

In [17]:
PE = positional_encoding(
    max_position=4,
    d_model=embedding_dim
)
print("Positional Encoding Shape:")
print(PE.shape)

print("\nPositional Encoding:")
print(PE.numpy())

Positional Encoding Shape:
(4, 8)

Positional Encoding:
[[ 0.0000000e+00  1.0000000e+00  0.0000000e+00  1.0000000e+00
   0.0000000e+00  1.0000000e+00  0.0000000e+00  1.0000000e+00]
 [ 8.4147096e-01  5.4030228e-01  9.9833414e-02  9.9500418e-01
   9.9998331e-03  9.9994999e-01  9.9999981e-04  9.9999952e-01]
 [ 9.0929741e-01 -4.1614684e-01  1.9866933e-01  9.8006660e-01
   1.9998666e-02  9.9980003e-01  1.9999987e-03  9.9999797e-01]
 [ 1.4112000e-01 -9.8999250e-01  2.9552022e-01  9.5533651e-01
   2.9995501e-02  9.9955004e-01  2.9999956e-03  9.9999553e-01]]


***ADDING WORD EMBEDDDINGS AND POSITIONAL ENCODING***

In [18]:
position_aware_embeddings = word_embeddings + PE[tf.newaxis, :]

print("Position Aware Embeddings Shape:")
print(position_aware_embeddings.shape)

print("\nPosition Aware Embeddings:")
print(position_aware_embeddings.numpy())

Position Aware Embeddings Shape:
(1, 4, 8)

Position Aware Embeddings:
[[[-0.01361709  1.0372405  -0.03849368  0.96881443  0.04991556
    0.97851837 -0.02508724  0.9915603 ]
  [ 0.8369405   0.5412988   0.14083217  1.0031633  -0.00433418
    0.9555297  -0.03699991  0.95178586]
  [ 0.8712313  -0.37794223  0.22344838  0.94261324  0.05697448
    0.9948705  -0.02469394  1.026215  ]
  [ 0.10790136 -0.9761139   0.32692614  0.9821205   0.06578732
    1.0248711  -0.04440983  0.98900914]]]


***MULTIHEAD ATTENTION***

In [19]:
attention_layer = MultiHeadAttention(
    num_heads=2,
    key_dim=embedding_dim
)

***APPLY SELF ATTENTION***

In [20]:
attention_output = attention_layer(
    query=position_aware_embeddings,
    key=position_aware_embeddings,
    value=position_aware_embeddings
)

print("Attention Output Shape:")
print(attention_output.shape)

print("\nContextualized Embeddings:")
print(attention_output.numpy())

Attention Output Shape:
(1, 4, 8)

Contextualized Embeddings:
[[[-0.22103086  0.03870117  0.34220862 -0.23378718 -0.54258615
    0.38522512 -0.150538   -0.14361982]
  [-0.2222895   0.037159    0.34029382 -0.23479988 -0.5425666
    0.3842897  -0.14977103 -0.14346837]
  [-0.22026943  0.04065097  0.3396396  -0.23597552 -0.54245126
    0.3835743  -0.14743869 -0.14374171]
  [-0.21652076  0.04644912  0.3400299  -0.23706691 -0.5424055
    0.38301408 -0.14443696 -0.14446005]]]


***FLOW SUMMARY***

In [21]:
print("Tokens Shape:", tokens.shape)
print("Word Embeddings Shape:", word_embeddings.shape)
print("Positional Encoding Shape:", PE.shape)
print("Position Aware Embeddings Shape:", position_aware_embeddings.shape)
print("Attention Output Shape:", attention_output.shape)

Tokens Shape: (1, 4)
Word Embeddings Shape: (1, 4, 8)
Positional Encoding Shape: (4, 8)
Position Aware Embeddings Shape: (1, 4, 8)
Attention Output Shape: (1, 4, 8)
